In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import scanpy as sc
import scvi
import sys
sys.path.append("../src/")
from multiHIVE.model import multiHIVE
import pandas as pd
import numpy as np

In [2]:
import torch, random
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)

In [ ]:
adata = sc.read('../Data/RNA_ADT/Stephenson/haniffa21.processed_healthy.h5ad')

In [4]:
sc.pp.normalize_total(adata ,target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
hvg = 4000

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=4000,
    flavor="seurat_v3",
    batch_key="batch",
    subset=True,
    layer="counts"
)

multiHIVE.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch",
    protein_expression_obsm_key="protein_counts"
)

INFO     Using column names from columns of adata.obsm['protein_counts']                                           
INFO     Found batches with missing protein expression                                                             


/tmp/ipykernel_1938755/2010409909.py:15: DeprecationWarning: multiHIVE is supposed to work with MuData. the use of anndata is deprecated and will be removed in scvi-tools 1.4. Please use setup_mudata
  multiHIVE.setup_anndata(


In [ ]:
vae = multiHIVE(adata,
                n_genes=adata.shape[1],
                n_regions=0,
                n_proteins=adata.obsm["protein_counts"].shape[1],
                mi_loss = True
                )
vae.train()
vae.get_latent_representation()

In [6]:
np.save("Stephenson.npy", adata.obsm['Z_multiHIVE'])